# When confidence is not enough

A score in $[0,1]$ is not a pipeline path and not seven derived guards. This notebook shows two ways people try to skip that, and what the calculator does.

First: high confidence with no packet. Second: leftover mass from an exact cochain solver, used as *one* guard. That solver measures leftover. It does not say where standing comes from.

Run from the package root, or from this `jupyter_notebooks/` folder.

In [1]:
from pathlib import Path
import sys

p = Path.cwd().resolve()
if p.name == "jupyter_notebooks":
    root = p.parent
elif (p / "code" / "coda_fda_core.py").is_file():
    root = p
elif (p / "coda_atlas" / "F005_coda" / "fda" / "code").is_dir():
    root = p / "coda_atlas" / "F005_coda" / "fda"
elif (p / "fda" / "code").is_dir():
    root = p / "fda"
else:
    root = p
assert (root / "code").is_dir(), "run from the package root or jupyter_notebooks/"
if str(root / "code") not in sys.path:
    sys.path.insert(0, str(root / "code"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from sympy import Matrix, Rational
from coda_fda_core import (
    DescentPacket,
    GuardEvidence,
    all_guards_true,
    derive_guards,
    evaluate_authority,
    evaluate_authorization,
    evidence_from_cochain_repair,
)

## Confidence without a packet

Supply green evidence and a confidence of $999/1000$. Omit the packet. The calculator refuses with `R_nopacket`. The score is accepted and ignored.

The compatibility helper that takes a pre-built all-true guard tuple does the same: still no packet, still no standing.

In [2]:
green = GuardEvidence(
    pipeline_accepted=True,
    replay_deterministic=True,
    residual_mass=Rational(0, 1),
    residual_zero_tol=Rational(0, 1),
    skeleton_preserved=True,
    candidates_checked=2,
    candidates_declared=2,
    checked_candidate_ids=("candidate_a", "candidate_b"),
    pending_lifts=0,
    active_refusal=False,
)
fluent = evaluate_authorization(
    "fluent_answer",
    None,
    green,
    logit_confidence=Rational(999, 1000),
)
print("no packet, high confidence:", fluent.inhabited, fluent.refusal_code)
assert fluent.inhabited is False
assert fluent.refusal_code == "R_nopacket"

also = evaluate_authority(
    "fluent_answer_2",
    None,
    all_guards_true(),
    logit_confidence=Rational(999, 1000),
)
print("pre-built green guards, still no packet:", also.inhabited, also.refusal_code)
assert also.inhabited is False
assert also.refusal_code == "R_nopacket"

no packet, high confidence: False R_nopacket
pre-built green guards, still no packet: False R_nopacket


## Leftover mass is one guard, not an origin theorem

Guard $g_3$ needs a leftover number. One lawful source is an exact cochain leftover: licensed repairs are multiples of $(1,-1)$, and leftover is ordinary $\ell^1$ distance to that line.

The vector $(2,-2)$ lies on the line, leftover $0$. The vector $(1,1)$ does not, leftover $2$. Feeding those numbers into $g_3$ can make standing inhabited or not. Computing leftover does not, by itself, say where standing comes from.

In [3]:
packet = DescentPacket(
    carrier_k="toy_carrier",
    evaluator_class_e=["candidate_a", "candidate_b"],
    verifier_v="replay_v1",
    terminalizer_tau="tau_standing",
    discipline_lambda="no_self_promote",
)
d0 = Matrix([[1], [-1]])

on_line, cert0 = evidence_from_cochain_repair(d0, Matrix([[2], [-2]]), checked_candidate_ids=("candidate_a", "candidate_b"))
print("on the repair line: leftover", cert0.distance)
assert cert0.distance == Rational(0)
assert derive_guards(on_line).zero_residual_mass is True
stood = evaluate_authorization("on_line", packet, on_line, residual_certificate=cert0)
print("standing inhabited?", stood.inhabited)
assert stood.inhabited is True

off_line, cert2 = evidence_from_cochain_repair(d0, Matrix([[1], [1]]), checked_candidate_ids=("candidate_a", "candidate_b"))
print("off the repair line: leftover", cert2.distance)
assert cert2.distance == Rational(2)
assert derive_guards(off_line).zero_residual_mass is False
refused = evaluate_authorization("off_line", packet, off_line, residual_certificate=cert2)
print("standing inhabited?", refused.inhabited, "refusal:", refused.refusal_code)
assert refused.inhabited is False
assert refused.refusal_code == "R_candidate"

on the repair line: leftover 0
standing inhabited? True
off the repair line: leftover 2
standing inhabited? False refusal: R_candidate
